# CySent Live RL Training — Qwen 2.5 3B

**REINFORCE with baseline** — trains the Qwen LLM directly against the live CySent environment.

| Detail | Value |
|--------|-------|
| Model | `Qwen/Qwen2.5-3B-Instruct` (3B, fp16) |
| Method | REINFORCE + EMA baseline, LoRA r=16 |
| Warm start | Optional SFT adapter from Unsloth training |
| Steps | ~30 episodes |
| GPU | T4 / L4 (Colab free/pro) |

Unlike the SFT notebook (which trains on a static dataset of state→action pairs),
this notebook makes the LLM **interact with the live environment**: observe → prompt → select action → env.step → reward → policy gradient update.

## How to run
1. Set `CYSENT_GIT_URL` below to your GitHub repo URL
2. (Optional) Set `HF_SFT_ADAPTER` to your Unsloth SFT adapter path for warm start
3. Run all cells. Training takes on the order of several hours on a T4 (30 episodes).
4. The final adapter is saved and can optionally be pushed to HF Hub / Google Drive.

In [ ]:
# Cell 1 — Install dependencies
!pip install -q --upgrade pip
!pip install -q torch --index-url https://download.pytorch.org/whl/cu121
!pip uninstall -y -q torchvision || true
!pip install -q --upgrade torchvision
!pip install -q transformers peft accelerate "bitsandbytes>=0.46.1"
# bitsandbytes required for 4-bit QLoRA on T4 / 16GB GPUs (auto-enabled in train_qwen_rl)
!pip install -q gymnasium numpy openenv stable-baselines3

import importlib.metadata as md
print('torch:', md.version('torch'))
print('torchvision:', md.version('torchvision'))
print('transformers:', md.version('transformers'))
print('peft:', md.version('peft'))
print('gymnasium:', md.version('gymnasium'))

import torch
import torchvision
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

torch: 2.11.0
torchvision: 0.26.0
transformers: 5.5.4
peft: 0.19.1
gymnasium: 1.2.3
CUDA available: False


In [3]:
# Cell 2 — Clone CySent repo and add to path
import os, sys, subprocess
from pathlib import Path

# >>> SET YOUR REPO URL HERE <<<
CYSENT_GIT_URL = os.environ.get('CYSENT_GIT_URL', 'https://github.com/sxchin-01/CySent.git')
cwd = Path.cwd()
if Path(cwd, 'backend', 'env', 'security_env.py').exists():
    DEFAULT_REPO_ROOT = str(cwd)
elif Path(cwd.parent, 'backend', 'env', 'security_env.py').exists():
    DEFAULT_REPO_ROOT = str(cwd.parent)
elif Path('/content').exists():
    DEFAULT_REPO_ROOT = '/content/CySent'
else:
    DEFAULT_REPO_ROOT = str(cwd / 'CySent')
REPO_ROOT = os.environ.get('CYSENT_REPO_PATH', DEFAULT_REPO_ROOT)

if not Path(REPO_ROOT, 'backend', 'env', 'security_env.py').exists():
    print(f'Cloning {CYSENT_GIT_URL} ...')
    result = subprocess.run(
        ['git', 'clone', '--depth', '1', CYSENT_GIT_URL, REPO_ROOT],
        text=True, capture_output=True, check=False,
    )
    if result.returncode != 0:
        print('STDERR:', result.stderr)
        raise RuntimeError('Clone failed. Set CYSENT_GIT_URL to your public repo.')
    print('Clone OK.')
else:
    print(f'CySent repo already present at: {REPO_ROOT}')

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

from backend.env.security_env import CySentSecurityEnv, ACTION_NAMES
print(f'Environment loaded. Actions: {len(ACTION_NAMES)}')
print('Actions:', list(ACTION_NAMES.values()))

CySent repo already present at: c:\programming\projects\CySent
Environment loaded. Actions: 12
Actions: ['do_nothing', 'patch_hr_systems', 'patch_web_server', 'patch_auth_server', 'rotate_credentials', 'isolate_suspicious_host', 'increase_monitoring', 'restore_backup', 'deploy_honeypot', 'phishing_training', 'investigate_top_alert', 'segment_finance_database']


In [4]:
# Cell 3 — Quick sanity check: run 5 random steps in the env
env = CySentSecurityEnv(max_steps=100, seed=42)
obs, info = env.reset()
print(f'Obs shape: {obs.shape}')
print(f'Initial risk: {info["network_risk"]:.3f}')

for i in range(5):
    action = env.action_space.sample()
    obs, reward, done, trunc, info = env.step(action)
    print(f'  Step {i+1}: action={ACTION_NAMES[action]}, reward={reward:.2f}, risk={info["network_risk"]:.3f}')
    if done or trunc:
        break
print('Env sanity check passed.')

Obs shape: (67,)
Initial risk: 0.228
  Step 1: action=segment_finance_database, reward=0.74, risk=0.228
  Step 2: action=patch_hr_systems, reward=1.30, risk=0.162
  Step 3: action=do_nothing, reward=3.33, risk=0.162
  Step 4: action=restore_backup, reward=0.58, risk=0.150
  Step 5: action=do_nothing, reward=2.03, risk=0.150
Env sanity check passed.


In [ ]:
# Cell 4 — Live RL Training (~30 episodes)
import os, sys, subprocess, re, urllib.request
from pathlib import Path
import torch

cwd = Path.cwd()
if Path(cwd, 'backend', 'train', 'train_qwen_rl.py').exists():
    DEFAULT_REPO_ROOT = str(cwd)
elif Path(cwd.parent, 'backend', 'train', 'train_qwen_rl.py').exists():
    DEFAULT_REPO_ROOT = str(cwd.parent)
elif Path('/content').exists():
    DEFAULT_REPO_ROOT = '/content/CySent'
else:
    DEFAULT_REPO_ROOT = str(cwd / 'CySent')

REPO_ROOT = os.environ.get('CYSENT_REPO_PATH', DEFAULT_REPO_ROOT)
CYSENT_GIT_URL = os.environ.get("CYSENT_GIT_URL", "https://github.com/sxchin-01/CySent.git")
CYSENT_GIT_BRANCH = os.environ.get("CYSENT_GIT_BRANCH", "main")
FORCE_REFRESH = os.environ.get("CYSENT_FORCE_REFRESH", "0") == "1"

p = Path(REPO_ROOT) / "backend" / "train" / "train_qwen_rl.py"
if FORCE_REFRESH or not p.exists():
    m = re.search(r"github\.com[:/]([^/]+)/([^/.]+)", CYSENT_GIT_URL)
    if m:
        user, repo = m.group(1), m.group(2)
        raw_url = (
            f"https://raw.githubusercontent.com/{user}/{repo}/{CYSENT_GIT_BRANCH}/"
            "backend/train/train_qwen_rl.py"
        )
        print(f"Downloading train_qwen_rl.py from {raw_url} ...")
        p.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(raw_url, p)
        print(f"  -> {p.stat().st_size:,} bytes")
    else:
        subprocess.run(["git", "-C", REPO_ROOT, "pull", "--ff-only"], check=False,
                       capture_output=True, text=True)
else:
    print(f"Using local train_qwen_rl.py at {p}")

# Force QLoRA on T4 (16 GB) — 4-bit base model fits, fp16 doesn't
os.environ["CYSENT_RL_QLORA"] = "1"
os.environ["CYSENT_RL_MAX_PROMPT_LEN"] = "256"

# Invalidate any cached old module
for mod_name in list(sys.modules):
    if "train_qwen_rl" in mod_name:
        del sys.modules[mod_name]

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

HF_SFT_ADAPTER = os.environ.get("HF_SFT_ADAPTER", None)
HF_TOKEN = os.environ.get("HF_TOKEN", None)

DEFAULT_OUTPUT_DIR = '/content/cysent_qwen_rl' if Path('/content').exists() else str(Path(REPO_ROOT) / 'outputs' / 'cysent_qwen_rl')
OUTPUT_DIR = os.environ.get("CYSENT_RL_OUTPUT_DIR", DEFAULT_OUTPUT_DIR)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

final_adapter_path = None
if not torch.cuda.is_available():
    print('No CUDA device detected in this runtime. Skipping live RL training cell.')
    print('Tip: run this cell on a CUDA runtime to execute full 30-step training.')
else:
    from backend.train.train_qwen_rl import train
    final_adapter_path = train(
        model_id="Qwen/Qwen2.5-3B-Instruct",
        adapter_path=HF_SFT_ADAPTER,
        token=HF_TOKEN,
        total_steps=30,
        max_turns=100,
        lr=1e-5,
        gamma=0.99,
        lora_r=16,
        lora_alpha=32,
        checkpoint_every=30,
        output_dir=OUTPUT_DIR,
        seed=42,
    )

print(f"Final adapter: {final_adapter_path}")

Using local train_qwen_rl.py at c:\programming\projects\CySent\backend\train\train_qwen_rl.py
No CUDA device detected in this runtime. Skipping live RL training cell.
Tip: run this cell on a CUDA runtime to execute full 50-step training.
Final adapter: None


In [10]:
# Cell 5 — Plot training curve
import json
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

history_path = Path(OUTPUT_DIR, 'training_history.json')
if not history_path.exists():
    print(f'No training history found at {history_path}. Skipping plot.')
else:
    history = json.loads(history_path.read_text())
    steps = [h['step'] for h in history]
    rewards = [h['reward'] for h in history]
    baselines = [h['baseline'] for h in history]

    # Smoothed reward (rolling average of 20)
    window = min(20, max(1, len(rewards)))
    smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(steps, rewards, alpha=0.3, color='cyan', label='Raw reward')
    ax1.plot(steps[window-1:], smoothed, color='orange', linewidth=2, label=f'Smoothed (w={window})')
    ax1.plot(steps, baselines, color='gray', linestyle='--', alpha=0.6, label='Baseline')
    ax1.set_xlabel('Training Step')
    ax1.set_ylabel('Episode Reward')
    ax1.set_title('Live RL Training — Qwen vs CySent Environment')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    losses = [h['loss'] for h in history]
    ax2.plot(steps, losses, color='red', alpha=0.5)
    ax2.set_xlabel('Training Step')
    ax2.set_ylabel('Policy Loss')
    ax2.set_title('REINFORCE Policy Loss')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'{OUTPUT_DIR}/training_curve.png', dpi=150, bbox_inches='tight')
    plt.show()

    # Summary stats
    half = max(1, len(rewards)//2)
    first_half = np.mean(rewards[:half])
    last_half = np.mean(rewards[-half:])
    print(f'Avg reward (first half): {first_half:.2f}')
    print(f'Avg reward (last  half): {last_half:.2f}')
    print(f'Improvement: {last_half - first_half:+.2f}')

No training history found at c:\programming\projects\CySent\outputs\cysent_qwen_rl\training_history.json. Skipping plot.


In [14]:
# Cell 6 — Benchmark: RL-trained Qwen vs PPO vs Random (10 episodes each)
import os
import sys
from pathlib import Path
import numpy as np
import torch
from backend.env.security_env import CySentSecurityEnv, ACTION_NAMES

REPO_ROOT = os.environ.get("CYSENT_REPO_PATH", "/content/CySent")
if not Path(REPO_ROOT).exists():
    REPO_ROOT = str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd())
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

def benchmark_llm(adapter_path, model_id='Qwen/Qwen2.5-3B-Instruct', episodes=10, max_turns=100, seed=42):
    """Run the RL-trained Qwen against the live env."""
    if not adapter_path or not Path(adapter_path).exists():
        print('No trained adapter found; skipping LLM benchmark.')
        return []

    # Import only when needed so CPU/no-training runs still execute this cell cleanly.
    from backend.train.train_qwen_rl import _build_prompt, ACTION_LIST
    from transformers import AutoModelForCausalLM, AutoTokenizer
    from peft import PeftModel

    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    base = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype, trust_remote_code=True)
    model = PeftModel.from_pretrained(base, adapter_path)
    model = model.to(device).eval()
    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    results = []
    for ep in range(episodes):
        env = CySentSecurityEnv(max_steps=max_turns, seed=seed + ep)
        obs, info = env.reset()
        total_reward = 0.0
        turns = 0
        while True:
            prompt = _build_prompt(info)
            inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            with torch.no_grad():
                out = model(**inputs)
            logits = out.logits[:, -1, :]
            token_ids = [tokenizer.encode(n, add_special_tokens=False)[0] for n in ACTION_LIST]
            action_idx = int(logits[0, token_ids].argmax().item())
            action_id = action_idx if action_idx < len(ACTION_NAMES) else 0
            obs, reward, done, trunc, info = env.step(action_id)
            total_reward += reward
            turns += 1
            if done or trunc:
                break
        results.append({'reward': total_reward, 'turns': turns, 'risk': info['network_risk']})
    return results

def benchmark_ppo(model_path, episodes=10, max_turns=100, seed=42):
    from stable_baselines3 import PPO
    model = PPO.load(model_path)
    results = []
    for ep in range(episodes):
        env = CySentSecurityEnv(max_steps=max_turns, seed=seed + ep)
        obs, info = env.reset()
        total_reward = 0.0
        turns = 0
        while True:
            action, _ = model.predict(obs, deterministic=True)
            obs, reward, done, trunc, info = env.step(int(action))
            total_reward += reward
            turns += 1
            if done or trunc:
                break
        results.append({'reward': total_reward, 'turns': turns, 'risk': info['network_risk']})
    return results

def benchmark_random(episodes=10, max_turns=100, seed=42):
    results = []
    for ep in range(episodes):
        env = CySentSecurityEnv(max_steps=max_turns, seed=seed + ep)
        obs, info = env.reset()
        total_reward = 0.0
        turns = 0
        while True:
            action = env.action_space.sample()
            obs, reward, done, trunc, info = env.step(action)
            total_reward += reward
            turns += 1
            if done or trunc:
                break
        results.append({'reward': total_reward, 'turns': turns, 'risk': info['network_risk']})
    return results

print('Benchmarking RL-trained Qwen...')
rl_results = benchmark_llm(final_adapter_path)

ppo_path = f'{REPO_ROOT}/backend/train/artifacts/cysent_ppo.zip'
if Path(ppo_path).exists():
    print('Benchmarking PPO...')
    ppo_results = benchmark_ppo(ppo_path)
else:
    print('PPO model not found, skipping PPO benchmark.')
    ppo_results = None

print('Benchmarking Random...')
rand_results = benchmark_random()

print('\n' + '='*65)
print(f'{"Agent":<22} {"Avg Reward":>12} {"Avg Turns":>12} {"Avg Risk":>12}')
print('='*65)

def _row(name, results):
    if not results:
        print(f'{name:<22} {"N/A":>12} {"N/A":>12} {"N/A":>12}')
        return
    r = np.mean([x['reward'] for x in results])
    t = np.mean([x['turns'] for x in results])
    k = np.mean([x['risk'] for x in results])
    print(f'{name:<22} {r:>12.2f} {t:>12.1f} {k:>12.3f}')

_row('Qwen RL (live)', rl_results)
if ppo_results:
    _row('PPO Defender', ppo_results)
_row('Random Baseline', rand_results)
print('='*65)

Benchmarking RL-trained Qwen...
No trained adapter found; skipping LLM benchmark.
Benchmarking PPO...
Benchmarking Random...

Agent                    Avg Reward    Avg Turns     Avg Risk
Qwen RL (live)                  N/A          N/A          N/A
PPO Defender                -156.66         90.7        0.391
Random Baseline               64.94         91.1        0.082


In [12]:
# Cell 7 — (Optional) Copy adapter to Google Drive
import shutil

# Uncomment and modify the destination:
# from google.colab import drive
# drive.mount('/content/drive')
# dst = '/content/drive/MyDrive/cysent_qwen_rl_adapter'
# shutil.copytree(final_adapter_path, dst, dirs_exist_ok=True)
# shutil.copy2(f'{OUTPUT_DIR}/training_history.json', f'{dst}/training_history.json')
# shutil.copy2(f'{OUTPUT_DIR}/training_curve.png', f'{dst}/training_curve.png')
# print(f'Adapter copied to {dst}')

In [ ]:
# Cell 8 — (Optional) Push adapter to Hugging Face Hub
# from huggingface_hub import HfApi
# api = HfApi(token=HF_TOKEN)
# api.upload_folder(
#     folder_path=final_adapter_path,
#     repo_id='YOUR_USERNAME/CySent-Qwen-RL-adapter',
#     repo_type='model',
#     commit_message='Qwen 2.5 3B live RL adapter (REINFORCE, 30 steps)',
# )
# print('Pushed to HF Hub.')